*0.4 Deep learning basics*

# BERT-style encoder fine-tuning

**The situation.** Every model so far was trained from zero on 8,000 sentences and plateaued around 78%. A model that has already read Wikipedia knows English; it only needs to learn *your* labels. Fine-tuning a pretrained encoder on a few thousand examples is how classification was done in production from 2019 until LLMs — and it still wins on cost per prediction.

**Fine-tuning.** Load a pretrained BERT with a fresh classification head, then run the training loop from item 4 on your labelled data with a small learning rate, for two or three epochs. The pretrained layers adjust slightly; the head learns from scratch. Here with Hugging Face's `Trainer`, which is the production version of that loop, on a tiny BERT (2 layers, 4M parameters) so it runs on a CPU in a few minutes.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

model_name = "google/bert_uncased_L-2_H-128_A-2"  # BERT-tiny: same architecture as BERT-base, 2 layers instead of 12
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2
)  # new 2-way head on top

sst2 = load_dataset("stanfordnlp/sst2")
train_rows = sst2["train"].shuffle(seed=0).select(range(8000))
val_rows = sst2["validation"]


def tokenize(rows):
    return tokenizer(rows["sentence"], truncation=True, max_length=64)


train_rows = train_rows.map(tokenize, batched=True)
val_rows = val_rows.map(tokenize, batched=True)


def compute_metrics(evaluation):
    predictions = np.argmax(evaluation.predictions, axis=1)
    return {"accuracy": float((predictions == evaluation.label_ids).mean())}


trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="/tmp/bert-tiny-sst2",
        learning_rate=5e-5,
        per_device_train_batch_size=32,
        num_train_epochs=3,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="no",
        report_to="none",
        use_cpu=True,
        seed=0,
    ),
    train_dataset=train_rows,
    eval_dataset=val_rows,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
before = trainer.evaluate()["eval_accuracy"]
print(f"before fine-tuning (random head): {before:.1%}")
trainer.train()
after = trainer.evaluate()["eval_accuracy"]
print(f"after 3 epochs: {after:.1%}")
assert after > before + 0.2

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

{'eval_loss': '0.6972', 'eval_model_preparation_time': '0.0003', 'eval_accuracy': '0.4977', 'eval_runtime': '0.2063', 'eval_samples_per_second': '4228', 'eval_steps_per_second': '528.5', 'epoch': 0}
before fine-tuning (random head): 49.8%


{'eval_loss': '0.5845', 'eval_model_preparation_time': '0.0003', 'eval_accuracy': '0.711', 'eval_runtime': '0.2072', 'eval_samples_per_second': '4208', 'eval_steps_per_second': '526', 'epoch': '1'}


{'eval_loss': '0.5121', 'eval_model_preparation_time': '0.0003', 'eval_accuracy': '0.7603', 'eval_runtime': '0.2279', 'eval_samples_per_second': '3826', 'eval_steps_per_second': '478.3', 'epoch': '2'}


{'eval_loss': '0.5032', 'eval_model_preparation_time': '0.0003', 'eval_accuracy': '0.7534', 'eval_runtime': '0.1971', 'eval_samples_per_second': '4423', 'eval_steps_per_second': '552.9', 'epoch': '3'}
{'train_runtime': '14.64', 'train_samples_per_second': '1640', 'train_steps_per_second': '51.24', 'train_loss': '0.5366', 'epoch': '3'}
{'eval_loss': '0.5032', 'eval_model_preparation_time': '0.0003', 'eval_accuracy': '0.7534', 'eval_runtime': '0.193', 'eval_samples_per_second': '4519', 'eval_steps_per_second': '564.8', 'epoch': '3'}
after 3 epochs: 75.3%


**Reading the output.** A random head scores around 50%. Three epochs later, a 2-layer, 4M-parameter BERT matches the best from-scratch model and beats the from-scratch transformer by ten points — because it arrived already knowing English. BERT-base (110M) on the same script reaches ~92% on this task; the pretraining did the hard part.

**Use it.** Two sentences through the fine-tuned model.

In [3]:
import torch

model.eval()
sentences = ["a warm, funny, and genuinely moving film", "two hours I will never get back"]
with torch.no_grad():
    logits = model(**tokenizer(sentences, return_tensors="pt", padding=True)).logits
for sentence, probability in zip(sentences, torch.softmax(logits, dim=1)[:, 1]):
    print(f"{probability:.0%} positive  {sentence}")
assert torch.softmax(logits, dim=1)[0, 1] > torch.softmax(logits, dim=1)[1, 1]

90% positive  a warm, funny, and genuinely moving film
22% positive  two hours I will never get back


**The rule to remember.** Do not train encoders from scratch; fine-tune a pretrained one. Small learning rate (2e-5 to 5e-5), 2–4 epochs, evaluate every epoch, keep the best.

| Use it when | Don't when | Instead use |
|---|---|---|
| a fixed classification/tagging task with ≥ 1k labels and high volume | no labels; task changes often; needs generation | an LLM (zero-shot or with a schema); use it to label data, then fine-tune this |

**Watch out**
- The learning rate is 100× smaller than from-scratch training; 1e-3 destroys the pretrained weights ("catastrophic forgetting").
- The `MISSING classifier.weight` warning at load time is expected — that is the new head. Any *other* missing weight is a wrong model name.
- Pin the model revision; hub models can change under the same name.